# Lesson 13 - Rock Paper Scissors Vision

In this lesson, SpiderPi uses MediaPipe hand recognition to play rock, paper, scissors with you.

## What We Will Do

1. Warm up the robot and camera
2. Detect a hand gesture
3. Turn the gesture into `rock`, `paper`, or `scissors`
4. Let SpiderPi count down: rock, paper, scissors, go
5. Reveal the robot move with the claw
6. Act out the result with lights, speech, and movement

In [ ]:
from lesson_loader import setup
setup()
from student_robot_v2 import bot
import random

bot.help()


In [ ]:
# Warm up SpiderPi for the game
bot.display.text("RPS Time", "Show a hand", "Rock Paper", "Scissors")
bot.display.smile()
bot.lights.blue()
bot.sound.beep(2200, 0.08)
bot.buzzer.off()
bot.display.text("Lets Play", "Rock Paper", "Scissors", "Show hand", seconds=1.0)
bot.camera.center_all()
bot.arm.open()


In [ ]:
# Detect a hand and inspect the result
hand_result = bot.vision.recognize_hands(show=True)
print(hand_result)


In [ ]:
# Pick the first game move that looks like rock, paper, or scissors
possible_moves = hand_result.get("game_moves", [])
player_move = possible_moves[0] if possible_moves else None
print("Player move:", player_move)

if player_move is None:
    bot.lights.yellow()
    bot.display.text("Try Again", "I could not", "see rock", "paper scissors")
    bot.display.text("Try Again", "Clear hand", "Rock Paper", "Scissors", seconds=1.0)
else:
    bot.display.text("Player Move", player_move.title(), "Nice!", "My turn next")
    bot.display.text("You Played", player_move.title(), seconds=1.0)


In [ ]:
# SpiderPi chooses a move, counts down, and decides who wins
robot_move = random.choice(["rock", "paper", "scissors"])
print("Robot move:", robot_move)

def show_robot_move(move):
    if move == "rock":
        bot.arm.close()
        bot.display.text("Spider Move", "Rock", "Claw closed", "Strong grip", seconds=0.8)
    elif move == "paper":
        bot.arm.open()
        bot.display.text("Spider Move", "Paper", "Claw open", "Flat hand", seconds=0.8)
    else:
        bot.arm.half_open()
        bot.display.text("Spider Move", "Scissors", "Half open", "Snip snip", seconds=0.8)

def countdown_and_show(move):
    beats = [
        ("Ready", 24, 12, 1100),
        ("Rock", 22, 12, 1300),
        ("Paper", 20, 12, 1500),
        ("Scissors", 18, 12, 1700),
        ("Go", 14, 10, 2100),
    ]
    bot.camera.center_all()
    bot.arm.open()
    for word, up_height, down_height, freq in beats:
        bot.display.text(word, seconds=0.7)
        bot.sound.beep(freq, 0.08)
        bot.arm.move(0, 15, up_height, seconds=0.18)
        bot.arm.move(0, 15, down_height, seconds=0.18)
    bot.buzzer.off()
    show_robot_move(move)

countdown_and_show(robot_move)

def winner(player, robot):
    if player is None:
        return "retry"
    if player == robot:
        return "draw"
    wins = {
        ("rock", "scissors"),
        ("paper", "rock"),
        ("scissors", "paper"),
    }
    return "player" if (player, robot) in wins else "robot"

result = winner(player_move, robot_move)
print("Result:", result)


In [ ]:
# Make SpiderPi react to the game result
if result == "retry":
    bot.lights.yellow()
    bot.display.text("No Score", "Need a clear", "hand sign", "Try again")
    bot.display.text("No Score", "Try Again", seconds=1.0)
    bot.sound.beep(900, 0.12)
    bot.sound.beep(700, 0.16)
elif result == "draw":
    bot.lights.purple()
    bot.display.text("Draw", f"You: {player_move}", f"Me: {robot_move}", "Same move!")
    bot.body.twist()
    bot.display.text("Draw", "Same move", seconds=1.0)
    bot.sound.beep(1200, 0.10)
    bot.sound.beep(1200, 0.10)
elif result == "player":
    bot.lights.red()
    bot.display.text("You Win", f"You: {player_move}", f"Me: {robot_move}", "Well played")
    bot.body.backward(0.4)
    bot.display.text("You Win", "Nice move", seconds=1.0)
    bot.sound.beep(1400, 0.08)
    bot.sound.beep(1800, 0.08)
    bot.sound.beep(2200, 0.14)
else:
    bot.lights.green()
    bot.display.text("I Win", f"You: {player_move}", f"Me: {robot_move}", "SpiderPi wins")
    bot.body.dance()
    bot.display.text("I Win", "SpiderPi wins", seconds=1.0)
    bot.sound.beep(2200, 0.08)
    bot.sound.beep(1800, 0.08)
    bot.sound.beep(1400, 0.14)

bot.buzzer.off()
bot.arm.home()
bot.body.stop()


## Challenge

Try one or more of these upgrades:

- change the countdown arm heights or speed
- make SpiderPi choose a special move sound for each result
- keep score across three rounds
- make different gestures trigger different action groups